# 01 - Data Loading & Schema Validation

Load the EIA Coal Consumption & Quality dataset and validate schema/dtypes.

In [1]:
import sys, os
ROOT = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), "src")) else os.path.dirname(os.getcwd())
sys.path.insert(0, ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
DATA = os.path.join(ROOT, "data", "eia_coal_consumption_quality_2020_2025.csv")
CHARTS = os.path.join(ROOT, "charts")
os.makedirs(CHARTS, exist_ok=True)
print("ROOT:", ROOT)


ROOT: C:\Users\Aditi\Documents\Projects\BDAA&SIG_hack_Fall26


In [2]:
def savefig(name):
    p = os.path.join(CHARTS, name)
    plt.savefig(p, dpi=150, bbox_inches='tight')
    print('saved', p)
    plt.show()


In [3]:
from src.data import load_coal_data, missingness_summary, MEASURE_COLUMNS, EXPECTED_COLUMNS
df = load_coal_data(DATA)
print('shape:', df.shape)
df.head()

shape: (1750, 19)


,period,location,stateDescription,sector,sectorDescription,ash-content,consumption,heat-content,price,receipts,stocks,sulfur-content,ash-content-units,consumption-units,heat-content-units,price-units,receipts-units,stocks-units,sulfur-content-units
0,2020,90,Pacific,1,Electric Utility,6.30,1541461.0,16.2558,42.25,873007.0,NaN,0.21,average ash percent by weight,short tons,average sulfur percent by weight,dollars per short ton,short tons,short tons,average sulfur percent by weight
1,2020,90,Pacific,2,IPP Non-CHP,8.36,3476488.0,16.6534,NaN,3525648.0,NaN,0.40,average ash percent by weight,short tons,average sulfur percent by weight,dollars per short ton,short tons,short tons,average sulfur percent by weight
2,2020,90,Pacific,3,IPP CHP,4.92,890674.0,18.6360,NaN,725275.0,NaN,0.27,average ash percent by weight,short tons,average sulfur percent by weight,dollars per short ton,short tons,short tons,average sulfur percent by weight
3,2020,90,Pacific,94,Independent Power Producers,7.70,4367162.0,17.0577,NaN,4250923.0,NaN,0.38,average ash percent by weight,short tons,average sulfur percent by weight,dollars per short ton,short tons,short tons,average sulfur percent by weight
4,2020,90,Pacific,98,Electric Power,7.48,5908623.0,16.8485,NaN,5123930.0,NaN,0.35,average ash percent by weight,short tons,average sulfur percent by weight,dollars per short ton,short tons,short tons,average sulfur percent by weight


In [4]:
df.dtypes

period                    int64
location                 object
stateDescription         object
sector                    int64
sectorDescription        object
ash-content             float64
consumption             float64
heat-content            float64
price                   float64
receipts                float64
stocks                  float64
sulfur-content          float64
ash-content-units        object
consumption-units        object
heat-content-units       object
price-units              object
receipts-units           object
stocks-units             object
sulfur-content-units     object
dtype: object

In [5]:
missingness_summary(df)

,missing,missing_pct
stocks,1215,69.43
price,1083,61.89
ash-content,192,10.97
sulfur-content,192,10.97
receipts,180,10.29
heat-content,27,1.54
consumption,15,0.86
stateDescription,0,0.00
location,0,0.00
period,0,0.00


In [6]:
print('Years:', sorted(df['period'].unique()))
print('Sectors:', sorted(df['sector'].unique()))
print('Distinct locations:', df['location'].nunique())

Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Sectors: [np.int64(1), np.int64(2), np.int64(3), np.int64(8), np.int64(9), np.int64(10), np.int64(94), np.int64(98)]
Distinct locations: 62


In [7]:
print('Sector codes & descriptions:')
print(df[['sector','sectorDescription']].drop_duplicates().sort_values('sector').to_string(index=False))

Sector codes & descriptions:
 sector            sectorDescription
      1             Electric Utility
      2                  IPP Non-CHP
      3                      IPP CHP
      8 Commercial and Institutional
      9                  Coke Plants
     10             Other Industrial
     94  Independent Power Producers
     98               Electric Power


In [8]:
# Validation assertions (mirror tests/test_data.py)
assert len(df) > 0
assert set(df['period'].unique()) == {2020,2021,2022,2023,2024,2025}
assert set(df['sector'].unique()) == {1,2,3,8,9,10,94,98}
assert df[MEASURE_COLUMNS].isna().any().any(), 'expected some blank (NaN) values'
assert (df[MEASURE_COLUMNS] == 0.0).any().any(), 'expected some reported 0.0 values'
print('All schema checks passed.')

All schema checks passed.


**Takeaways**
- 1,750 rows across 62 locations x 8 sectors x 6 years.
- Seven numeric measures; blanks become NaN while reported `0.0` (e.g. California IPP rows) is preserved.
- 2025 data is preliminary and sparse (visible in the missingness report).